# tensor-wraps-ndarray composite — cx22: slice-view mutation on a from_numpy tensor propagates to the ndarray

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `slice-view-mutation`, `tensor-wraps-ndarray`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "tensor-wraps-ndarray"
DD_ATOM_IDS = ["slice-view-mutation", "tensor-wraps-ndarray"]
DD_SUBTOPICS = ["PyTorch: Slice view mutation", "PyTorch: tensor from ndarray"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

`t.from_numpy(arr)` wraps an ndarray WITHOUT copying — the tensor and the array share the same underlying storage. Slice-view mutation says that assigning to `tensor[...] = value` mutates the source storage, not just a local copy. Compose them and you get: writing to a slice of a from_numpy tensor mutates the ORIGINAL ndarray, visible from the numpy side. The reverse is also true: mutating `arr` shows up in the tensor.

Contrast with `t.tensor(arr)` which COPIES — mutations don't propagate. The composite drill verifies both directions of the from_numpy aliasing AND that `t.tensor` is isolated.

### Composite Exercise — slice-view mutation on a from_numpy tensor propagates to the ndarray

**Atoms exercised together**: `slice-view-mutation`, `tensor-wraps-ndarray`

Implement `cx22_aliasing_writes(arr)` that:

1. Build `wrapped = t.from_numpy(arr)` (shares storage) and `copied = t.tensor(arr)` (independent copy).
2. Write `wrapped[0] = 999.0` — a slice-view mutation that must propagate to `arr` (through the from_numpy aliasing).
3. Write `arr[1] = 777.0` — a numpy-side mutation that must propagate back into `wrapped` (but NOT into `copied`).
4. Return a dict reporting `arr_after`, `wrapped_after`, `copied_after`, `wrapped_shares_storage`, `copied_shares_storage`.

The contract: from_numpy aliases (mutations propagate both ways), t.tensor does not.

In [ ]:
def cx22_aliasing_writes(arr):
    # tensor-wraps-ndarray: from_numpy shares storage; t.tensor copies.
    wrapped = t.from_numpy(arr)
    copied = t.tensor(arr)

    # slice-view mutation through the alias: wrapped[0]=999 hits arr's storage.
    wrapped[0] = 999.0
    # And the reverse: numpy-side mutation is visible in wrapped (same storage).
    arr[1] = 777.0

    # Storage-sharing detection: from_numpy shares; t.tensor does not.
    wrapped_shares = wrapped.data_ptr() == arr.__array_interface__['data'][0]
    copied_shares = copied.data_ptr() == arr.__array_interface__['data'][0]
    return {
        'arr_after': arr.copy(),
        'wrapped_after': wrapped.clone(),
        'copied_after': copied.clone(),
        'wrapped_shares_storage': bool(wrapped_shares),
        'copied_shares_storage': bool(copied_shares),
    }


<details><summary>Show solution — cx22</summary>

```python
def cx22_aliasing_writes(arr):
    # tensor-wraps-ndarray: from_numpy shares storage; t.tensor copies.
    wrapped = t.from_numpy(arr)
    copied = t.tensor(arr)

    # slice-view mutation through the alias: wrapped[0]=999 hits arr's storage.
    wrapped[0] = 999.0
    # And the reverse: numpy-side mutation is visible in wrapped (same storage).
    arr[1] = 777.0

    # Storage-sharing detection: from_numpy shares; t.tensor does not.
    wrapped_shares = wrapped.data_ptr() == arr.__array_interface__['data'][0]
    copied_shares = copied.data_ptr() == arr.__array_interface__['data'][0]
    return {
        'arr_after': arr.copy(),
        'wrapped_after': wrapped.clone(),
        'copied_after': copied.clone(),
        'wrapped_shares_storage': bool(wrapped_shares),
        'copied_shares_storage': bool(copied_shares),
    }
```

The two atoms collapse into one mental model: from_numpy creates an ALIAS, t.tensor creates a SNAPSHOT. Slice-view writes (`wrapped[0]=999`) just exercise that alias — they're not special, they're the normal in-place semantics of a view. If you wanted the tensor to be independent, you'd write `t.tensor(arr)` (or `t.from_numpy(arr).clone()`). Don't use `from_numpy` if the source ndarray might mutate under you.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx22'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx22',
        'subtopics': ["PyTorch: Slice view mutation", "PyTorch: tensor from ndarray"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()